In [119]:
import pandas as pd
import numpy as np
import json

## Define the metadata

In [120]:
organism = "homo_sapiens"
cell_source = "adipose"
cell_state = None

reference = "GRCh38"
paper_doi = "https://www.nature.com/articles/s41467-023-36983-2#Sec34"
table_link = "https://static-content.springer.com/esm/art%3A10.1038%2Fs41590-021-00922-4/MediaObjects/41590_2021_922_MOESM2_ESM.xlsx"

# don't include in header
table_name = "41467_2023_36983_MOESM4_ESM.xlsx"
table_name = "paper_degs.xlsx" # local name 

## Read in file

In [121]:
excel = pd.read_excel(table_name, skiprows = 0, sheet_name=["17_Angueira et al."])

df = pd.concat(excel.values())
ctmap = pd.read_csv("ctmap/ctmap_angueira.txt", sep="\t", header=None, names=["ctnum", "ctname"], index_col=0)
df["cluster_name"] = df["cluster"].map(ctmap["ctname"])

df

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,cluster_name
0,EEF1A1,4.024568e-206,0.660106,0.745,0.551,1.194129e-201,0,Adipocyte
1,IGHG3,3.443877e-38,0.430059,0.169,0.097,1.021833e-33,0,Adipocyte
2,FABP4,1.217300e-110,0.414764,0.844,0.586,3.611851e-106,0,Adipocyte
3,B2M,6.344955e-24,0.386782,0.434,0.434,1.882612e-19,0,Adipocyte
4,FTL,1.086667e-18,0.364124,0.254,0.210,3.224249e-14,0,Adipocyte
...,...,...,...,...,...,...,...,...
46173,ARL15,8.855273e-01,-0.379113,0.759,0.702,1.000000e+00,23,smooth muscle
46174,ANO6,9.737803e-01,-0.180530,0.777,0.666,1.000000e+00,23,smooth muscle
46175,PID1,9.791153e-01,-0.218497,0.402,0.384,1.000000e+00,23,smooth muscle
46176,MYO5A,9.924217e-01,-0.176698,0.348,0.322,1.000000e+00,23,smooth muscle


In [122]:
df.head()

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,cluster_name
0,EEF1A1,4.024568e-206,0.660106,0.745,0.551,1.194129e-201,0,Adipocyte
1,IGHG3,3.443877e-38,0.430059,0.169,0.097,1.021833e-33,0,Adipocyte
2,FABP4,1.217300e-110,0.414764,0.844,0.586,3.611851e-106,0,Adipocyte
3,B2M,6.344955e-24,0.386782,0.434,0.434,1.882612e-19,0,Adipocyte
4,FTL,1.086667e-18,0.364124,0.254,0.210,3.224249e-14,0,Adipocyte


## Unfiltered

In [123]:
unfiltered_df = df.copy()
unfiltered_df.rename(columns ={"cluster_name": "cell_type_label", "gene": "feature_name"}, inplace=True)
unfiltered_df["organism"] = organism
unfiltered_df["cell_source"] = cell_source
unfiltered_df["cell_state"] = cell_state
unfiltered_df["feature_identifier"] = None

unfiltered_df.drop(['p_val', 'avg_log2FC', 'pct.1', 'pct.2', 'p_val_adj'], axis=1, inplace=True) 
result = [] 

for _, row in unfiltered_df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "unfiltered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'Adipocyte',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'EEF1A1',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'Adipocyte',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'EEF1A1',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'unfiltered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'Adipocyte',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG3',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'Adipocyte',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'IGHG3',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   '

In [124]:
with open('evidence_unfiltered_angueira.json', 'w') as f:
    json.dump(result, f, indent = 4)

## Perform the filtering

In [125]:
col_pval = "p_val_adj"
col_lfc = "avg_log2FC"
col_gene = "gene"
col_ct = "cluster_name" # says cluster in table, but before this had "celltype"
col_rank = "pct.1" # set to None if pct. 1DNE

In [126]:
min_mean = 100
max_pval = 1e-10
min_lfc = 2.2
max_gene_shares = 2
max_per_celltype = 20

# filter by pval and lfc
dfc = df.query(f"{col_pval} <= {max_pval} & {col_lfc} >= {min_lfc}")

# mask out genes that are shared between max_gene_shares cell types
non_repeat_genes = dfc[col_gene].value_counts()[dfc[col_gene].value_counts() < max_gene_shares].index.values

m = dfc[dfc[col_gene].isin(non_repeat_genes)]

if col_rank:
    m = m.sort_values(col_rank, ascending = True)

# max number to sample is equal to the min number of genes across all celltype
n_sample = min(m[col_ct].value_counts().max(), max_per_celltype)

# sample n_sample genes
markers = m.groupby(col_ct).tail(n_sample)
markers_dict = markers.groupby(col_ct)[col_gene].apply(lambda x: list(x)).to_dict()

## Check how many genes per cell type

In [127]:
markers

,gene,p_val,avg_log2FC,pct.1,pct.2,p_val_adj,cluster,cluster_name
30518,NKAIN2,1.624020e-203,2.494707,0.370,0.063,4.818629e-199,16,Endo
42728,ADGRL3,1.224136e-122,2.325794,0.556,0.075,3.632134e-118,22,smooth muscle
28795,FAM155A,0.000000e+00,2.204404,0.635,0.123,0.000000e+00,15,Endo
16612,AL357507.1,0.000000e+00,2.349605,0.663,0.080,0.000000e+00,8,Endo
44535,PCDH7,1.026978e-95,2.214234,0.679,0.107,3.047145e-91,23,smooth muscle
44539,KCNMA1,6.509471e-74,2.439450,0.705,0.152,1.931425e-69,23,smooth muscle
42711,CRISPLD2,2.674090e-160,2.557265,0.712,0.097,7.934292e-156,22,smooth muscle
21703,GPAT3,0.000000e+00,2.441945,0.715,0.132,0.000000e+00,11,Adipocyte
44526,LMOD1,1.608442e-158,2.214490,0.732,0.079,4.772408e-154,23,smooth muscle
42690,RCAN2,2.180997e-294,2.308611,0.739,0.057,6.471236e-290,22,smooth muscle


In [128]:
markers.groupby(col_ct)[col_rank].mean().sort_values()

cluster_name
smooth muscle    0.789231
Endo             0.796273
Adipocyte        0.865333
M2 Macrophage    0.903167
Fibroblast       0.960950
Name: pct.1, dtype: float64

In [129]:
markers[col_ct].value_counts()

cluster_name
Fibroblast       20
smooth muscle    13
Endo             11
M2 Macrophage     6
Adipocyte         3
Name: count, dtype: int64

## Convert to evidence json


In [130]:
df = markers[[col_ct, col_gene]].rename(columns={col_ct : "cell_type_label", col_gene: "feature_name"})
df["organism"] = organism
df["cell_source"] = cell_source
df["cell_state"] = cell_state
df["feature_identifier"] = df["feature_name"]
# df["feature_identifier"] = df["feature_identifier"].apply(run)
df["feature_identifier"] = None
save = df.to_dict(orient = "records")

result = [] 

for _, row in df.iterrows():
    result.append({"extracted": {"organism": row["organism"], "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene"},
                   "derived": {"organism": row["organism"], "cell_type_id": None, "cell_type_label": row["cell_type_label"], "cell_source": row["cell_source"], "cell_state": row["cell_state"], "feature_name": row["feature_name"], "feature_type": "gene", "feature_identifier": row["feature_identifier"], "feature_type": "ensembl"},
                   "source": {"source_type": "deg", "source_rationale": "filtered", "source_id": "deg"}})

result

[{'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'Endo',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'NKAIN2',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'Endo',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'NKAIN2',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'source_rationale': 'filtered',
   'source_id': 'deg'}},
 {'extracted': {'organism': 'homo_sapiens',
   'cell_type_label': 'smooth muscle',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'ADGRL3',
   'feature_type': 'gene'},
  'derived': {'organism': 'homo_sapiens',
   'cell_type_id': None,
   'cell_type_label': 'smooth muscle',
   'cell_source': 'adipose',
   'cell_state': None,
   'feature_name': 'ADGRL3',
   'feature_type': 'ensembl',
   'feature_identifier': None},
  'source': {'source_type': 'deg',
   'so

## Save evidence

In [131]:
with open("evidence_angueira.json", "w") as f:
    json.dump(result, f, indent = 4) 